# License Plate Detection — Model 1: YOLOv8s
**CMPS 261 — Machine Learning Project**

YOLOv8s is a single-stage detector that predicts bounding boxes and class probabilities in one forward pass — very fast and accurate.

This notebook runs **both locally and on Google Colab**. On Colab, select Runtime → Change runtime type → T4 GPU for fast training.

## 1. Environment Setup

Auto-detects Colab vs local and configures paths accordingly.

In [ ]:
import sys, os

YOLO_IMGSZ = 960  # Higher resolution helps small plate localization; use 640 for faster baseline reruns.

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    # Install dependencies
    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', '-q'], check=True)

    # Mount Drive and extract dataset
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/yolo'):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Extracted dataset.')
    else:
        print('Dataset already extracted.')

    # Write dataset YAML with Colab absolute paths
    yaml_content = """path: /content/data/yolo
train: images/train
val:   images/val
test:  images/test
nc: 1
names: ['licence']
"""
    yaml_path = '/content/data/yolo/dataset.yaml'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    MODEL_DIR   = '/content/models'
    RESULTS_DIR = '/content/results'
    WEIGHTS_OUT = f'/content/models/yolov8s_imgsz{YOLO_IMGSZ}/weights/best.pt'
    WEIGHTS_SAVE = '/content/yolov8s_best.pt'
    TEST_IMG_DIR = '/content/data/yolo/images/test'
    DEVICE = 0  # GPU index

else:
    sys.path.append('..')
    from src.prepare_data import prepare
    yaml_path = prepare()

    import torch
    DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

    MODEL_DIR   = '../models'
    RESULTS_DIR = '../results'
    WEIGHTS_OUT  = f'../models/yolov8s_imgsz{YOLO_IMGSZ}/weights/best.pt'
    WEIGHTS_SAVE = '../models/yolov8s_best.pt'
    TEST_IMG_DIR = '../data/yolo/images/test'

os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Dataset YAML : {yaml_path}')
print(f'Device       : {DEVICE}')

## 2. Train YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data     = yaml_path,
    epochs   = 100,
    imgsz    = YOLO_IMGSZ,
    batch    = 16,
    device   = DEVICE,
    project  = MODEL_DIR,
    name     = f'yolov8s_imgsz{YOLO_IMGSZ}',
    exist_ok = True,
    verbose  = True,
)

## 3. Evaluate on Test Set

In [ ]:
test_metrics = model.val(split='test', imgsz=YOLO_IMGSZ)
yolo_f1 = 2 * test_metrics.box.mp * test_metrics.box.mr / (test_metrics.box.mp + test_metrics.box.mr + 1e-6)
print(f'Test mAP@0.5      : {test_metrics.box.map50:.4f}')
print(f'Test mAP@0.5:0.95 : {test_metrics.box.map:.4f}')
print(f'Test Precision    : {test_metrics.box.mp:.4f}')
print(f'Test Recall       : {test_metrics.box.mr:.4f}')
print(f'Test F1           : {yolo_f1:.4f}')

## 4. Save Best Weights

In [ ]:
import shutil
shutil.copy(WEIGHTS_OUT, WEIGHTS_SAVE)
print(f'Saved weights to: {WEIGHTS_SAVE}')

## 5. Visualise Predictions on Test Images

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

test_images  = random.sample(os.listdir(TEST_IMG_DIR), 8)
model_best   = YOLO(WEIGHTS_SAVE)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, fname in zip(axes, test_images):
    img_path = os.path.join(TEST_IMG_DIR, fname)
    result   = model_best.predict(img_path, conf=0.25, verbose=False)[0]
    img      = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor='lime', facecolor='none'))
        ax.text(x1, y1-4, f'{conf:.2f}', color='lime', fontsize=8,
                bbox=dict(facecolor='black', alpha=0.4, pad=1))
    ax.set_title(fname, fontsize=7); ax.axis('off')

plt.suptitle('YOLOv8s — Predictions on Test Set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'yolo_predictions.png'), dpi=150)
plt.show()
print(f'Saved: {RESULTS_DIR}/yolo_predictions.png')

## 6. Save Metrics

In [ ]:
import json

yolo_metrics = {
    'model'    : 'YOLOv8s',
    'epochs'   : 100,
    'imgsz'    : YOLO_IMGSZ,
    'map50'    : round(test_metrics.box.map50, 4),
    'map50_95' : round(test_metrics.box.map,   4),
    'precision': round(test_metrics.box.mp,    4),
    'recall'   : round(test_metrics.box.mr,    4),
    'f1'       : round(float(yolo_f1), 4),
}
metrics_path = os.path.join(RESULTS_DIR, 'yolo_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(yolo_metrics, f, indent=2)
print(json.dumps(yolo_metrics, indent=2))

## 7. Download Weights (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(WEIGHTS_SAVE)
    files.download(metrics_path)
    print('Downloading weights and metrics...')
else:
    print(f'Weights saved at: {WEIGHTS_SAVE}')